# Working logic for converting from nifti predictions to dicom for upload to arterys/tempus

In [ ]:

import os
import pandas as pd
from pathlib import Path
import SimpleITK as sitk
import shutil
import numpy as np
import pydicom
from pydicom.uid import ExplicitVRLittleEndian, generate_uid
from datetime import datetime
import zipfile


patient = 'Balboloop'
dataset = 'all_patients'
dicom_catalog_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/dicom_catalog_4d-flow_{patient}.csv"
dicom_catalog = pd.read_csv(dicom_catalog_path)

def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

out_ogs_dir = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/dicoms/ogs"
out_vse_dir = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/dicoms/vse"
reset_dir(out_ogs_dir)
reset_dir(out_vse_dir)

timepoint = 0
new_study_instance_uid = generate_uid()
new_series_instance_uid_mag = generate_uid()
new_series_instance_uid_v3 = generate_uid()
new_series_instance_uid_v4 = generate_uid()
new_series_instance_uid_v5 = generate_uid()

for timepoint in range(20):
    prediction_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/predictions/pred_{patient}_devoted-frost-50_epoch-630_all_timepoints_t{timepoint:02d}_overlap_16_overlap-mode_hann.nii.gz"
    dicom_catalog_tp = dicom_catalog[(dicom_catalog['time_index'] == timepoint) &
                                    (dicom_catalog['tag_0x0043_0x1030'].isin([2, 3, 4, 5]))].copy()
    
    dicom_catalog_tp = dicom_catalog_tp.sort_values('slice_index').reset_index(drop=True)

    dicom_catalog_tp_mg = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 2].copy()
    dicom_catalog_tp_v3 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 3].copy()
    dicom_catalog_tp_v4 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 4].copy()
    dicom_catalog_tp_v5 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 5].copy()
    
    dicom_filepaths = [Path(row['filepath']) for index, row in dicom_catalog_tp_mg.iterrows()]
    dicom_filepaths_v3 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v3.iterrows()]
    dicom_filepaths_v4 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v4.iterrows()]
    dicom_filepaths_v5 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v5.iterrows()]
    # for fp in dicom_filepaths:
    #     print(fp.name)
        
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(fp) for fp in dicom_filepaths])
    dicom_3d = reader.Execute()

    prediction_img = sitk.ReadImage(prediction_path)

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(dicom_3d)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(0.0)

    resampled_img = resampler.Execute(prediction_img)

    dicom_np = sitk.GetArrayFromImage(dicom_3d)
    pred_np = sitk.GetArrayFromImage(resampled_img)

    dicom_low, dicom_high = np.percentile(dicom_np, [1, 99.9])
    pred_low, pred_high = np.percentile(pred_np, [1, 99.9])

    pred_clipped = np.clip(pred_np, pred_low, pred_high)
    mapped = (pred_clipped - pred_low) / (pred_high - pred_low)
    mapped = mapped * (dicom_high - dicom_low) + dicom_low
    mapped_int16 = np.rint(mapped).astype(np.int16)

    for z, dcm_path in enumerate(dicom_filepaths):
        # og = pydicom.dcmread(dcm_path)
        dv3_path = dicom_filepaths_v3[z]
        dv4_path = dicom_filepaths_v4[z]
        dv5_path = dicom_filepaths_v5[z]
        
        
        dcm = pydicom.dcmread(dcm_path)
        dcm_v3 = pydicom.dcmread(dv3_path)
        dcm_v4 = pydicom.dcmread(dv4_path)
        dcm_v5 = pydicom.dcmread(dv5_path)
        
        
        if dcm.file_meta.TransferSyntaxUID.is_compressed:
            dcm.decompress()
            dcm.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
            dcm.is_little_endian = True
            # dcm.is_implicit_VR = False
        
        slice_array = mapped_int16[z,:,:]
        assert slice_array.shape == (dcm.Rows, dcm.Columns)
        
        assert dcm.BitsAllocated == 16
        assert dcm.BitsStored == 16
        assert dcm.HighBit == 15
        assert dcm.PixelRepresentation == 1
        
        dcm.PixelData = slice_array.tobytes()
        
        def update_dcm(dcm,study_uid, series_uid, offset=0):
            dcm.ImageType = r"DERIVED\SECONDARY"
            dcm.StudyInstanceUID = study_uid
            dcm.SeriesInstanceUID = series_uid
            dcm.SOPInstanceUID = generate_uid()
            dcm.SeriesDescription = f"{dcm.SeriesDescription} VSE"
            dcm.PatientID = patient
            dcm.SeriesNumber = 9000 + (dcm.SeriesNumber % 100 + offset)
            dcm.ContentDate = datetime.now().strftime('%Y%m%d')
            dcm.ContentTime = datetime.now().strftime('%H%M%S.%f')[:-3]
        
        update_dcm(dcm, study_uid=new_study_instance_uid, series_uid=new_series_instance_uid_mag)
        update_dcm(dcm_v3, study_uid=new_study_instance_uid, series_uid=new_series_instance_uid_v3, offset=1)
        update_dcm(dcm_v4, study_uid=new_study_instance_uid, series_uid=new_series_instance_uid_v4, offset=2)
        update_dcm(dcm_v5, study_uid=new_study_instance_uid, series_uid=new_series_instance_uid_v5, offset=3)
        
        dcm.save_as(Path(out_vse_dir) / f"{dcm_path.stem}_vse.dcm", write_like_original=False)
        dcm_v3.save_as(Path(out_vse_dir) / f"{dv3_path.stem}_v3.dcm", write_like_original=False)
        dcm_v4.save_as(Path(out_vse_dir) / f"{dv4_path.stem}_v4.dcm", write_like_original=False)
        dcm_v5.save_as(Path(out_vse_dir) / f"{dv5_path.stem}_v5.dcm", write_like_original=False)
        
        shutil.copy2(dcm_path, Path(out_ogs_dir) / dcm_path.name)
        # og.save_as(Path(out_ogs_dir) / f"{dcm_path.stem}_ogs.dcm", write_like_original=False)

zip_path = Path(out_vse_dir).parent / f"{patient}_dicom_predictions.zip"
# remove zip file if it exists
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all files in output_dir to the zip
    for file_path in Path(out_vse_dir).rglob('*'):
        if file_path.is_file():
            # Use relative path from output_dir to maintain directory structure
            arcname = file_path.relative_to(out_vse_dir)
            zipf.write(file_path, arcname=arcname)

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281

ImageSeriesReader (0x560d7acf6ad

# set patient name and dicom path

In [59]:
import os
import pandas as pd

patient = 'Balboloop'
dataset = 'all_patients'
dicom_catalog_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/dicom_catalog_4d-flow_{patient}.csv"
dicom_catalog = pd.read_csv(dicom_catalog_path)
display(dicom_catalog.head())

,filepath,patientid,studyinstanceuid,seriesinstanceuid,seriesnumber,sopinstanceuid,modality,studydate,seriesdescription,instancenumber,...,acquisitionnumber,imagesinacquisition,stackid,instackpositionnumber,slicelocation,locationsinacquisition,num3dslabs,locsper3dslab,time_index,slice_index
0,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.100000403025816259866829127508564906572,MR,NaN,NaN,2094,...,1,140,1.0,1047.0,97.6542,140,1,140,13,104
1,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.100425264043067745442831225933913660278,MR,NaN,NaN,2280,...,1,140,1.0,1140.0,113.8540,140,1,140,19,113
2,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.100464071225147266881515490302390543225,MR,NaN,NaN,2182,...,1,140,1.0,1091.0,106.6540,140,1,140,1,109
3,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.100575612005496622481321789559575884067,MR,NaN,NaN,2336,...,1,140,1.0,1168.0,119.2540,140,1,140,15,116
4,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.100633646498046960845989160341780468051,MR,NaN,NaN,1682,...,1,140,1.0,841.0,61.6541,140,1,140,1,84


# Set timepoint and slice the dicom catalog accordingly

In [60]:
timepoint = 0
dicom_catalog_tp = dicom_catalog[(dicom_catalog['time_index'] == timepoint) &
                                (dicom_catalog['tag_0x0043_0x1030'].isin([2, 3, 4, 5]))].copy()
display(dicom_catalog_tp.head())

,filepath,patientid,studyinstanceuid,seriesinstanceuid,seriesnumber,sopinstanceuid,modality,studydate,seriesdescription,instancenumber,...,acquisitionnumber,imagesinacquisition,stackid,instackpositionnumber,slicelocation,locationsinacquisition,num3dslabs,locsper3dslab,time_index,slice_index
12,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.101388890619244198931671931011123405052,MR,NaN,NaN,821,...,1,140,1.0,411.0,-15.74590,140,1,140,0,41
25,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.102118524902904757156886763664133044479,MR,NaN,NaN,1101,...,1,140,1.0,551.0,9.45415,140,1,140,0,55
42,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.104115019831868209757683101889697777106,MR,NaN,NaN,661,...,1,140,1.0,331.0,-30.14580,140,1,140,0,33
51,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.104792719150152288840902167779967532141,MR,NaN,NaN,1641,...,1,140,1.0,821.0,58.05410,140,1,140,0,82
71,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.20365087898108142913317662564411020584,4232,2.25.107225626624901636507303025849409019826,MR,NaN,NaN,2101,...,1,140,1.0,1051.0,99.45420,140,1,140,0,105


In [ ]:
dicom_catalog_tp_mg = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 2].copy()
display(dicom_catalog_tp_mg.head())

,filepath,patientid,studyinstanceuid,seriesinstanceuid,seriesnumber,sopinstanceuid,modality,studydate,seriesdescription,instancenumber,...,acquisitionnumber,imagesinacquisition,stackid,instackpositionnumber,slicelocation,locationsinacquisition,num3dslabs,locsper3dslab,time_index,slice_index
8440,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.104584911361387202882988387654037368041,MR,NaN,NaN,1521,...,1,140,1.0,761.0,47.2541,140,1,140,0,76
8450,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.106081930979131942115491250512773193255,MR,NaN,NaN,2641,...,1,140,1.0,1321.0,148.0540,140,1,140,0,132
8543,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.116849590818950744396320170141822620690,MR,NaN,NaN,2701,...,1,140,1.0,1351.0,153.4540,140,1,140,0,135
8555,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.118102433918844615133466982235725662590,MR,NaN,NaN,2101,...,1,140,1.0,1051.0,99.4542,140,1,140,0,105
8592,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.121663615042493253805459535097421649052,MR,NaN,NaN,1601,...,1,140,1.0,801.0,54.4541,140,1,140,0,80


In [ ]:
# sort by slice_index
dicom_catalog_tp_mg = dicom_catalog_tp_mg.sort_values('slice_index').reset_index(drop=True)
display(dicom_catalog_tp_mg.head())


,filepath,patientid,studyinstanceuid,seriesinstanceuid,seriesnumber,sopinstanceuid,modality,studydate,seriesdescription,instancenumber,...,acquisitionnumber,imagesinacquisition,stackid,instackpositionnumber,slicelocation,locationsinacquisition,num3dslabs,locsper3dslab,time_index,slice_index
0,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.222382276330659217078158387874537788454,MR,NaN,NaN,1,...,1,140,1.0,1.0,-89.5459,140,1,140,0,0
1,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.291899669696589496914699914335336232407,MR,NaN,NaN,21,...,1,140,1.0,11.0,-87.7459,140,1,140,0,1
2,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.206038103611923806746798380559450943408,MR,NaN,NaN,41,...,1,140,1.0,21.0,-85.9459,140,1,140,0,2
3,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.158548441224865982793114299554876698432,MR,NaN,NaN,61,...,1,140,1.0,31.0,-84.1459,140,1,140,0,3
4,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.83474229762169234758435775158552767010,2.25.271800506227515547636850293047168622256,4230,2.25.78321229395767703517988375102095855198,MR,NaN,NaN,81,...,1,140,1.0,41.0,-82.3459,140,1,140,0,4


In [63]:
for index, row in dicom_catalog_tp_mg.iterrows():
    print(index,row['filepath'])


0 /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Balboloop/2.25.83474229762169234758435775158552767010/2.25.271800506227515547636850293047168622256/2.25.222382276330659217078158387874537788454.dcm
1 /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Balboloop/2.25.83474229762169234758435775158552767010/2.25.271800506227515547636850293047168622256/2.25.291899669696589496914699914335336232407.dcm
2 /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Balboloop/2.25.83474229762169234758435775158552767010/2.25.271800506227515547636850293047168622256/2.25.206038103611923806746798380559450943408.dcm
3 /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Balboloop/2.25.83474229762169234758435775158552767010/2.25.271800506227515547636850293047168622256/2.25.158548441224865982793114299554876698432.dcm
4 /h

# obtain dicom filepaths for series

In [64]:
from pathlib import Path

dicom_filepaths = [Path(row['filepath']) for index, row in dicom_catalog_tp_mg.iterrows()]
for fp in dicom_filepaths:
    print(fp.name)

2.25.222382276330659217078158387874537788454.dcm
2.25.291899669696589496914699914335336232407.dcm
2.25.206038103611923806746798380559450943408.dcm
2.25.158548441224865982793114299554876698432.dcm
2.25.78321229395767703517988375102095855198.dcm
2.25.53992079025712475460810566775810294445.dcm
2.25.199669607559424452127803655955819226541.dcm
2.25.162914313646071349020294098282177068022.dcm
2.25.271368764566088398415872156958797740383.dcm
2.25.309819180295895023719285168492486239447.dcm
2.25.220015635001453127071270233836417273809.dcm
2.25.202715144240262067803636436526414297322.dcm
2.25.240503346656399039401247445731239500849.dcm
2.25.188242583807802040617171295972488284460.dcm
2.25.229807082364913630736239047243744314043.dcm
2.25.63458884846003662351634406374831803680.dcm
2.25.209518428856933659433401096891962362820.dcm
2.25.328511401124084163157628752158245914694.dcm
2.25.328624074987674078691901090426355544313.dcm
2.25.196140650380526735971637473427545043143.dcm
2.25.852129090865481630

In [65]:
import pydicom

for fp in dicom_filepaths:
    dicom_ds = pydicom.dcmread(fp)
    print("PatientID:", dicom_ds.PatientID)
    print("SeriesInstanceUID:", dicom_ds.SeriesInstanceUID)
    print("SOPInstanceUID:", dicom_ds.SOPInstanceUID)
    print("SeriesNumber:", dicom_ds.SeriesNumber)
    print("SeriesDescription:", dicom_ds.SeriesDescription)
    
    print("BitsAllocated:", dicom_ds.BitsAllocated)
    print("BitsStored:", dicom_ds.BitsStored)
    print("HighBit:", dicom_ds.HighBit)
    print("PixelRepresentation:", dicom_ds.PixelRepresentation)
    
    arr = dicom_ds.pixel_array
    print("PixelData:", arr.dtype)
    print("PixelData shape:", arr.shape)
    print("PixelData size:", arr.size)
    print("PixelData nbytes:", arr.nbytes)
    print("PixelData itemsize:", arr.itemsize)
    print("PixelData ndim:", arr.ndim)
    print("PixelData size:", arr.size)
    
    break

    
    
    

PatientID: 
SeriesInstanceUID: 2.25.271800506227515547636850293047168622256
SOPInstanceUID: 2.25.222382276330659217078158387874537788454
SeriesNumber: 4230
SeriesDescription: 
BitsAllocated: 16
BitsStored: 16
HighBit: 15
PixelRepresentation: 1
PixelData: int16
PixelData shape: (256, 256)
PixelData size: 65536
PixelData nbytes: 131072
PixelData itemsize: 2
PixelData ndim: 2
PixelData size: 65536


# Read dicom series from filepaths obtained from dicom catalog

## read the dicom

In [66]:
import SimpleITK as sitk

reader = sitk.ImageSeriesReader()
reader.SetFileNames([str(fp) for fp in dicom_filepaths])
dicom_3d = reader.Execute()

ImageSeriesReader (0x560d7acf6ad0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199281



### read a single dicom

In [67]:
single_dicom = sitk.ReadImage(dicom_filepaths[0])

all_keys = list(single_dicom.GetMetaDataKeys())
print(f"\nMetadata keys available: {len(all_keys)}")

for k in all_keys:
    print(f"{k}: {single_dicom.GetMetaData(k)}")



Metadata keys available: 109
0008|0000: 314
0008|0005: ISO_IR 192
0008|0008: ORIGINAL\PRIMARY\OTHER
0008|0016: 1.2.840.10008.5.1.4.1.1.4
0008|0018: 2.25.222382276330659217078158387874537788454
0008|0020: 
0008|0021: 
0008|0022: 
0008|0023: 
0008|0030: 
0008|0031: 
0008|0032: 
0008|0033: 
0008|0050: 
0008|0060: MR
0008|0070: GE MEDICAL SYSTEMS
0008|0080: 
0008|0090: 
0008|1010: 
0008|1030: 
0008|103e: 
0008|1070: 
0008|1090: DISCOVERY MR750 
0010|0000: 56
0010|0010: 
0010|0020: 
0010|0030: 
0010|0040: 
0010|1010: 
0010|1030: 
0010|21b0: 
0012|0062: YES 
0012|0063: BASIC APPLICATION LEVEL CONFIDENTIALITY PROFILE 
0018|0000: 496
0018|0010: 
0018|0020: RM
0018|0021: NONE
0018|0022: CG\EDR_GEMS\VASCTOF_GEMS\ACC_GEMS 
0018|0023: 3D
0018|0025: N 
0018|0050: 3.59
0018|0080: 4.939 
0018|0081: 2.564 
0018|0082: 0 
0018|0083: 1 
0018|0084: 127.802000
0018|0085: 1H
0018|0086: 1 
0018|0087: 3 
0018|0088: 3.59
0018|0091: 1 
0018|0093: 100 
0018|0094: 70
0018|0095: 488.281 
0018|1000: 
0018|1020: 30

In [68]:
ds = pydicom.dcmread(dicom_filepaths[0], stop_before_pixels=False)
print(ds.file_meta.TransferSyntaxUID)
print(ds.file_meta.TransferSyntaxUID.is_compressed)



1.2.840.10008.1.2.4.90
True


In [69]:
ds.decompress()
print("test decompression successful")

test decompression successful


## affine from sitk

In [70]:
import numpy as np

print(type(dicom_3d))
print(dicom_3d.GetSize())
print(dicom_3d.GetSpacing())
print(dicom_3d.GetOrigin())
# print(dicom_3d.GetDirection())
D = np.array(dicom_3d.GetDirection()).reshape(3,3)
print(D)

spacing = np.array(dicom_3d.GetSpacing())
origin = np.array(dicom_3d.GetOrigin())

A = np.eye(4)
A[:3,:3] = D @ np.diag(spacing)
A[:3, 3] = origin

print(A)

<class 'SimpleITK.SimpleITK.Image'>
(256, 256, 140)
(1.4063, 1.4063, 1.7999992805755396)
(-158.866, -148.215, -89.5459)
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[   1.4063        0.            0.         -158.866     ]
 [   0.            1.4063        0.         -148.215     ]
 [   0.            0.            1.79999928  -89.5459    ]
 [   0.            0.            0.            1.        ]]


## other info

In [71]:
print(dicom_3d.GetPixelID())
print(dicom_3d.GetPixelIDTypeAsString())
print(dicom_3d.GetPixelIDValue())
print(dicom_3d.GetNumberOfComponentsPerPixel())


2
16-bit signed integer
2
1


In [72]:
arr_3d = sitk.GetArrayFromImage(dicom_3d)

print("Array shape:", arr_3d.shape)
print("Array dtype:", arr_3d.dtype)
print("Array size:", arr_3d.size)
print("Array nbytes:", arr_3d.nbytes)
print("Array itemsize:", arr_3d.itemsize)
print("Array ndim:", arr_3d.ndim)
print("Array size:", arr_3d.size)
print("Array max:", arr_3d.max())
print("Array min:", arr_3d.min())
print("Array mean:", arr_3d.mean())
print("Array 1st percentile:", np.percentile(arr_3d, 1))
print("Array 99th percentile:", np.percentile(arr_3d, 99))
print("Array 95th percentile:", np.percentile(arr_3d, 95))
print("Array 2nd percentile:", np.percentile(arr_3d, 2))
print("Array 98th percentile:", np.percentile(arr_3d, 98))



Array shape: (140, 256, 256)
Array dtype: int16
Array size: 9175040
Array nbytes: 18350080
Array itemsize: 2
Array ndim: 3
Array size: 9175040
Array max: 4925
Array min: 0
Array mean: 404.2602820260184


Array 1st percentile: 0.0
Array 99th percentile: 2598.0
Array 95th percentile: 1690.0
Array 2nd percentile: 0.0
Array 98th percentile: 2291.0


## visualize dicom

import matplotlib.pyplot as plt

for z in range(arr_3d.shape[0]):
    plt.imshow(arr_3d[z,:,:], cmap='gray')
    plt.show()


# Pull the consolidated 4d flow nifti for this patient

In [74]:
nifti_consolidated_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/nifti/4d_flow_mag_{patient}.nii.gz"
print(nifti_consolidated_path)
nifti_consolidated_img = sitk.ReadImage(nifti_consolidated_path)
print(nifti_consolidated_img)

# pull nifti for that patient and timepoint

/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/4d_flow_mag_Balboloop.nii.gz
Image (0x560d7ac930b0)
  RTTI typeinfo:   itk::Image<short, 4u>
  Reference Count: 1
  Modified Time: 14125
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 14098
  UpdateMTime: 14121
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 140, 20]
  BufferedRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 140, 20]
  RequestedRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 140, 20]
  Spacing: [1.40625, 1.40625, 1.79987, 1]
  Origin: [-158.866, -148.215, -89.5459, 0]
  Direction: 
1 0 0 0
0 1 0 0
0 0 1 0
0 0 0 1

  IndexToPointMatrix: 
1.40625 0 0 0
0 1.40625 0 0
0 0 1.79987 0
0 0 0 1

  PointToIndexMatrix: 
0.711111 0 0 

# Pull the 3d cine niftis for that patient just to check the affines

## consolidated 3d cine

In [75]:
nifti_cine_consolidated_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/nifti/3d_cine_{patient}.nii.gz"
print(nifti_cine_consolidated_path)
nifti_cine_consolidated_img = sitk.ReadImage(nifti_cine_consolidated_path)
print(nifti_cine_consolidated_img)

/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/3d_cine_Balboloop.nii.gz
Image (0x560d79eb3070)
  RTTI typeinfo:   itk::Image<short, 4u>
  Reference Count: 1
  Modified Time: 14363
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 14335
  UpdateMTime: 14359
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 70, 20]
  BufferedRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 70, 20]
  RequestedRegion: 
    Dimension: 4
    Index: [0, 0, 0, 0]
    Size: [256, 256, 70, 20]
  Spacing: [1.4844, 1.4844, 1.8, 1]
  Origin: [-172.763, -162.297, -71.0991, 0]
  Direction: 
0.999843 0.00327 0.0174088 0
-0.00304001 0.99991 -0.013095 0
-0.0174501 0.01304 0.999763 0
0 0 0 1

  IndexToPointMatrix: 
1.48416 0.00485399 0.0313359 0
-0

## single timepoint

In [76]:
nifti_cine_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/nifti/3d_cine_{patient}_per_timepoint/3d_cine_{patient}_frame_{timepoint:02d}.nii.gz"
print(nifti_cine_path)
nifti_cine_img = sitk.ReadImage(nifti_cine_path)
print(nifti_cine_img)


/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/3d_cine_Balboloop_per_timepoint/3d_cine_Balboloop_frame_00.nii.gz
Image (0x560d7a8e41e0)
  RTTI typeinfo:   itk::Image<short, 3u>
  Reference Count: 1
  Modified Time: 14584
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 14558
  UpdateMTime: 14580
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  Spacing: [1.4844, 1.4844, 1.8]
  Origin: [-172.763, -162.297, -71.0991]
  Direction: 
0.999843 0.0032689 0.0174088
-0.00304111 0.99991 -0.013095
-0.0174501 0.01304 0.999763

  IndexToPointMatrix: 
1.48416 0.00485235 0.0313359
-0

# Pull 4d flow nifti for that patient and timepoint

In [77]:
nifti_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/{dataset}/patient_data/{patient}/nifti/4d_flow_mag_{patient}_per_timepoint/4d_flow_mag_{patient}_frame_{timepoint:02d}.nii.gz"
print(nifti_path)

/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/4d_flow_mag_Balboloop_per_timepoint/4d_flow_mag_Balboloop_frame_00.nii.gz


In [78]:
nifti_img = sitk.ReadImage(nifti_path)
print(nifti_img)

Image (0x560d7b067620)
  RTTI typeinfo:   itk::Image<short, 3u>
  Reference Count: 1
  Modified Time: 14805
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 14779
  UpdateMTime: 14801
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 70]
  Spacing: [1.4844, 1.4844, 1.8]
  Origin: [-172.763, -162.297, -71.0991]
  Direction: 
0.999843 0.0032689 0.0174088
-0.00304111 0.99991 -0.013095
-0.0174501 0.01304 0.999763

  IndexToPointMatrix: 
1.48416 0.00485235 0.0313359
-0.00451421 1.48427 -0.023571
-0.0259028 0.0193566 1.79957

  PointToIndexMatrix: 
0.673569 -0.00204872 -0.0117557
0.00220217 0.673612 0.0087847
0.00967159 -0.00727502 0.

# pull prediction for that patient and timepoint

In [79]:
prediction_path = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/predictions/pred_{patient}_devoted-frost-50_epoch-630_all_timepoints_t{timepoint:02d}_overlap_16_overlap-mode_hann.nii.gz"
print(prediction_path)

/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/Balboloop/predictions/pred_Balboloop_devoted-frost-50_epoch-630_all_timepoints_t00_overlap_16_overlap-mode_hann.nii.gz


In [80]:
prediction_img = sitk.ReadImage(prediction_path)
print(prediction_img)


Image (0x560d7b045610)
  RTTI typeinfo:   itk::Image<float, 3u>
  Reference Count: 1
  Modified Time: 15026
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 15000
  UpdateMTime: 15022
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [272, 272, 90]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [272, 272, 90]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [272, 272, 90]
  Spacing: [1.4, 1.4, 1.4]
  Origin: [-172.808, -162.336, -71.2988]
  Direction: 
0.999843 0.0032689 0.0174088
-0.00304111 0.99991 -0.013095
-0.0174501 0.01304 0.999763

  IndexToPointMatrix: 
1.39978 0.00457646 0.0243724
-0.00425755 1.39987 -0.018333
-0.0244301 0.018256 1.39967

  PointToIndexMatrix: 
0.714174 -0.00217222 -0.0124643
0.00233493 0.714221 0.0093143
0.0124349 -0.00935358 0.714116



In [81]:
print(type(prediction_img))
print(prediction_img.GetSize())
print(prediction_img.GetSpacing())
print(prediction_img.GetOrigin())
# print(dicom_3d.GetDirection())
D = np.array(prediction_img.GetDirection()).reshape(3,3)
print(D)

spacing = np.array(dicom_3d.GetSpacing())
origin = np.array(dicom_3d.GetOrigin())

A = np.eye(4)
A[:3,:3] = D @ np.diag(spacing)
A[:3, 3] = origin

print(A)

<class 'SimpleITK.SimpleITK.Image'>
(272, 272, 90)
(1.399999976158142, 1.399999976158142, 1.399999976158142)
(-172.8084716796875, -162.33615112304688, -71.29883575439453)
[[ 0.99984311  0.0032689   0.01740884]
 [-0.00304111  0.99990963 -0.01309501]
 [-0.01745007  0.01304002  0.9997627 ]]
[[ 1.40607937e+00  4.59705029e-03  3.13358938e-02 -1.58866000e+02]
 [-4.27671354e-03  1.40617292e+00 -2.35710117e-02 -1.48215000e+02]
 [-2.45400291e-02  1.83381736e-02  1.79957214e+00 -8.95459000e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [82]:
print(prediction_img.GetPixelID())
print(prediction_img.GetPixelIDTypeAsString())
print(prediction_img.GetPixelIDValue())
print(prediction_img.GetNumberOfComponentsPerPixel())

8
32-bit float
8
1


In [83]:
prd_3d = sitk.GetArrayFromImage(prediction_img)

print("Array shape:", prd_3d.shape)
print("Array dtype:", prd_3d.dtype)
print("Array size:", prd_3d.size)
print("Array nbytes:", prd_3d.nbytes)
print("Array itemsize:", prd_3d.itemsize)
print("Array ndim:", prd_3d.ndim)
print("Array size:", prd_3d.size)
print("Array max:", prd_3d.max())
print("Array min:", prd_3d.min())
print("Array mean:", prd_3d.mean())
print("Array 1st percentile:", np.percentile(prd_3d, 1))
print("Array 99th percentile:", np.percentile(prd_3d, 99))
print("Array 95th percentile:", np.percentile(prd_3d, 95))


Array shape: (90, 272, 272)
Array dtype: float32
Array size: 6658560
Array nbytes: 26634240
Array itemsize: 4
Array ndim: 3
Array size: 6658560
Array max: 0.85498905
Array min: 3.832715e-06
Array mean: 0.05094913
Array 1st percentile: 0.000113200942723779
Array 99th percentile: 0.3921957153081894
Array 95th percentile: 0.21068901121616362


## visualize the prediction 

import matplotlib.pyplot as plt

for z in range(prd_3d.shape[0]):
    plt.imshow(prd_3d[z,:,:], cmap='gray')
    plt.show()


# resample the prediction into the dicom space

In [85]:
import SimpleITK as sitk

fixed_img = dicom_3d
moving_img = prediction_img

resampler = sitk.ResampleImageFilter()
resampler.SetReferenceImage(fixed_img)
resampler.SetInterpolator(sitk.sitkLinear)
resampler.SetTransform(sitk.Transform())
resampler.SetDefaultPixelValue(0.0)


resampled_img = resampler.Execute(moving_img)

print(resampled_img)



Image (0x560d7acd21b0)
  RTTI typeinfo:   itk::Image<float, 3u>
  Reference Count: 1
  Modified Time: 15081
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 15064
  UpdateMTime: 15080
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 140]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 140]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [256, 256, 140]
  Spacing: [1.4063, 1.4063, 1.8]
  Origin: [-158.866, -148.215, -89.5459]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
1.4063 0 0
0 1.4063 0
0 0 1.8

  PointToIndexMatrix: 
0.711086 0 0
0 0.711086 0
0 0 0.555556

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (0x560d7b4ee360)
      RTTI typeinfo:   itk::ImportImageContainer<unsigned long, float

In [86]:
print(resampled_img.GetPixelID())
print(resampled_img.GetPixelIDTypeAsString())
print(resampled_img.GetPixelIDValue())
print(resampled_img.GetNumberOfComponentsPerPixel())

8
32-bit float
8
1


In [87]:
resampled_3d = sitk.GetArrayFromImage(resampled_img)

print("Array shape:", resampled_3d.shape)
print("Array dtype:", resampled_3d.dtype)
print("Array size:", resampled_3d.size)
print("Array nbytes:", resampled_3d.nbytes)
print("Array itemsize:", prd_3d.itemsize)
print("Array ndim:", prd_3d.ndim)
print("Array size:", prd_3d.size)
print("Array max:", resampled_3d.max())
print("Array min:", resampled_3d.min())
print("Array mean:", resampled_3d.mean())
print("Array 1st percentile:", np.percentile(resampled_3d, 1))
print("Array 99th percentile:", np.percentile(resampled_3d, 99))
print("Array 95th percentile:", np.percentile(resampled_3d, 95))
print("Array 2nd percentile:", np.percentile(resampled_3d, 2))
print("Array 98th percentile:", np.percentile(resampled_3d, 98))


Array shape: (140, 256, 256)
Array dtype: float32
Array size: 9175040
Array nbytes: 36700160
Array itemsize: 4
Array ndim: 3
Array size: 6658560
Array max: 0.8388211
Array min: 0.0
Array mean: 0.028136585
Array 1st percentile: 0.0
Array 99th percentile: 0.33222225576639053
Array 95th percentile: 0.1429719313979146
Array 2nd percentile: 0.0
Array 98th percentile: 0.2502387011051185


## visualize resampling

import matplotlib.pyplot as plt

for z in range(resampled_3d.shape[0]):
    plt.imshow(resampled_3d[z,:,:], cmap='gray')
    plt.show()

# convert to int16

In [89]:
dicom_np = sitk.GetArrayFromImage(dicom_3d)
pred_np = sitk.GetArrayFromImage(resampled_img)

dicom_low, dicom_high = np.percentile(dicom_np, [1, 99])
pred_low, pred_high = np.percentile(pred_np, [1, 99])

print(dicom_low, dicom_high)
print(pred_low, pred_high)

0.0 2598.0
0.0 0.33222225576639053


In [90]:
pred_clipped = np.clip(pred_np, pred_low, pred_high)
print(pred_clipped.min(), pred_clipped.max())

0.0 0.33222225


In [91]:
mapped = (pred_clipped - pred_low) / (pred_high - pred_low)
mapped = mapped * (dicom_high - dicom_low) + dicom_low
print(mapped.min(), mapped.max())


0.0 2598.0


In [92]:
mapped_int16 = np.rint(mapped).astype(np.int16)
print(mapped_int16.min(), mapped_int16.max())


0 2598


In [93]:
from pathlib import Path
import shutil

def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

out_ogs_dir = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/dicoms/ogs"
out_vse_dir = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/{patient}/dicoms/vse"
reset_dir(out_ogs_dir)
reset_dir(out_vse_dir)



In [94]:
from pydicom.uid import ExplicitVRLittleEndian, generate_uid
import shutil
from datetime import datetime
reset_dir(out_ogs_dir)
reset_dir(out_vse_dir)

new_series_instance_uid = generate_uid()

for z, dcm_path in enumerate(dicom_filepaths):
    # og = pydicom.dcmread(dcm_path)
    dcm = pydicom.dcmread(dcm_path)
    
    if dcm.file_meta.TransferSyntaxUID.is_compressed:
        dcm.decompress()
        dcm.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
        dcm.is_little_endian = True
        # dcm.is_implicit_VR = False
    
    slice_array = mapped_int16[z,:,:]
    assert slice_array.shape == (dcm.Rows, dcm.Columns)
    
    assert dcm.BitsAllocated == 16
    assert dcm.BitsStored == 16
    assert dcm.HighBit == 15
    assert dcm.PixelRepresentation == 1
    
    dcm.PixelData = slice_array.tobytes()
    
    dcm.ImageType = "VSE"
    dcm.SeriesInstanceUID = new_series_instance_uid
    dcm.SOPInstanceUID = generate_uid()
    dcm.SeriesDescription = f"{dcm.SeriesDescription} VSE"
    dcm.SeriesNumber = 9000 + (dcm.SeriesNumber % 100)
    dcm.ContentDate = datetime.now().strftime('%Y%m%d')
    dcm.ContentTime = datetime.now().strftime('%H%M%S.%f')[:-3]
    dcm.PatientID = patient
    
    dcm.save_as(Path(out_vse_dir) / f"{dcm_path.stem}_vse.dcm", write_like_original=False)
    shutil.copy2(dcm_path, Path(out_ogs_dir) / dcm_path.name)
    # og.save_as(Path(out_ogs_dir) / f"{dcm_path.stem}_ogs.dcm", write_like_original=False)




In [119]:
dicom_catalog_tp = dicom_catalog[(dicom_catalog['time_index'] == timepoint) &
                                    (dicom_catalog['tag_0x0043_0x1030'].isin([2, 3, 4, 5]))].copy()
    
dicom_catalog_tp = dicom_catalog_tp.sort_values('slice_index').reset_index(drop=True)

dicom_catalog_tp_mg = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 2].copy()
# dicom_catalog_tp_mg = dicom_catalog_tp_mg.sort_values('slice_index').reset_index(drop=True)
dicom_catalog_tp_v3 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 3].copy()
# dicom_catalog_tp_v3 = dicom_catalog_tp_v3.sort_values('slice_index').reset_index(drop=True)
dicom_catalog_tp_v4 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 4].copy()
# dicom_catalog_tp_v4 = dicom_catalog_tp_v4.sort_values('slice_index').reset_index(drop=True)
dicom_catalog_tp_v5 = dicom_catalog_tp[dicom_catalog_tp['tag_0x0043_0x1030'] == 5].copy()
# dicom_catalog_tp_v5 = dicom_catalog_tp_v5.sort_values('slice_index').reset_index(drop=True)

dicom_filepaths = [Path(row['filepath']) for index, row in dicom_catalog_tp_mg.iterrows()]
dicom_filepaths_v3 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v3.iterrows()]
dicom_filepaths_v4 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v4.iterrows()]
dicom_filepaths_v5 = [Path(row['filepath']) for index, row in dicom_catalog_tp_v5.iterrows()]


for z, dcm_path, dcm_path_v3, dcm_path_v4, dcm_path_v5 in zip(range(len(dicom_filepaths)), dicom_filepaths, dicom_filepaths_v3, dicom_filepaths_v4, dicom_filepaths_v5):
    dcm = pydicom.dcmread(dcm_path)
    dcm_v3 = pydicom.dcmread(dcm_path_v3)
    dcm_v4 = pydicom.dcmread(dcm_path_v4)
    dcm_v5 = pydicom.dcmread(dcm_path_v5)
    
    
    
    print(dcm.InstanceNumber)
    print(dcm_v3.InstanceNumber)
    print(dcm_v4.InstanceNumber)
    print(dcm_v5.InstanceNumber)
    
    
    

20
20
20
20
40
40
40
40
60
60
60
60
80
80
80
80
100
100
100
100
120
120
120
120
140
140
140
140
160
160
160
160
180
180
180
180
200
200
200
200
220
220
220
220
240
240
240
240
260
260
260
260
280
280
280
280
300
300
300
300
320
320
320
320
340
340
340
340
360
360
360
360
380
380
380
380
400
400
400
400
420
420
420
420
440
440
440
440
460
460
460
460
480
480
480
480
500
500
500
500
520
520
520
520
540
540
540
540
560
560
560
560
580
580
580
580
600
600
600
600
620
620
620
620
640
640
640
640
660
660
660
660
680
680
680
680
700
700
700
700
720
720
720
720
740
740
740
740
760
760
760
760
780
780
780
780
800
800
800
800
820
820
820
820
840
840
840
840
860
860
860
860
880
880
880
880
900
900
900
900
920
920
920
920
940
940
940
940
960
960
960
960
980
980
980
980
1000
1000
1000
1000
1020
1020
1020
1020
1040
1040
1040
1040
1060
1060
1060
1060
1080
1080
1080
1080
1100
1100
1100
1100
1120
1120
1120
1120
1140
1140
1140
1140
1160
1160
1160
1160
1180
1180
1180
1180
1200
1200
1200
1200
1220
1220
12

In [126]:
print(len(dicom_filepaths))
print(len(dicom_filepaths_v3))
print(len(dicom_filepaths_v4))
print(len(dicom_filepaths_v5))


140
140
140
140


In [128]:
print(dcm_path.stem)
print(dcm_path_v3.stem)
print(dcm_path_v4.stem)
print(dcm_path_v5.stem)

2.25.145597126863394274295208214528501837861
2.25.264804617837755903578242652803270293748
2.25.95048143545339902638101459531064397100
2.25.180483070688799209297470426980528135274


In [125]:
print(Path(out_vse_dir) / f"{dcm_path_v5.stem}_v5.dcm")

#check if file exists

print(os.path.exists(Path(out_vse_dir) / f"{dcm_path_v5.stem}_v5.dcm"))



/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/Balboloop/dicoms/vse/2.25.180483070688799209297470426980528135274_v5.dcm
True


# misc

In [11]:
import os
import pandas as pd

dicom_catalog_path = "/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Boudubat/dicom_catalog_4d-flow_Boudubat.csv"
dicom_catalog = pd.read_csv(dicom_catalog_path)
dicom_catalog.head()

,filepath,patientid,studyinstanceuid,seriesinstanceuid,seriesnumber,sopinstanceuid,modality,studydate,seriesdescription,instancenumber,...,acquisitionnumber,imagesinacquisition,stackid,instackpositionnumber,slicelocation,locationsinacquisition,num3dslabs,locsper3dslab,time_index,slice_index
0,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.52479434339957415477335998526887329855,2.25.21310654792733449088383413317637630180,4183,2.25.100043257890946166978030362472113806990,MR,NaN,NaN,1572,...,1,140,1.0,786.0,112.8,140,1,140,11,78
1,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.52479434339957415477335998526887329855,2.25.21310654792733449088383413317637630180,4183,2.25.10010450492345777604836427088095054970,MR,NaN,NaN,2524,...,1,140,1.0,1262.0,199.2,140,1,140,3,126
2,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.52479434339957415477335998526887329855,2.25.21310654792733449088383413317637630180,4183,2.25.100162749333940625613677506327878481932,MR,NaN,NaN,914,...,1,140,1.0,457.0,53.4,140,1,140,13,45
3,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.52479434339957415477335998526887329855,2.25.21310654792733449088383413317637630180,4183,2.25.100359623832464375717594231922815134744,MR,NaN,NaN,862,...,1,140,1.0,431.0,49.8,140,1,140,1,43
4,/home/ayeluru/mnt/fourier/repository/vascular-...,NaN,2.25.52479434339957415477335998526887329855,2.25.21310654792733449088383413317637630180,4183,2.25.10048150312528654907580322351254263332,MR,NaN,NaN,2210,...,1,140,1.0,1105.0,170.4,140,1,140,9,110


In [12]:

prediction_path = "/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/Boudubat/predictions"
output_dir = "/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/devoted-frost-50_epoch-630_all_timepoints/Boudubat/dicom_predictions"
patient_id = "Boudubat"
print(sorted(os.listdir(prediction_path)))

['pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t00_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t01_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t02_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t03_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t04_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t05_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t06_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t07_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t08_overlap_16_overlap-mode_hann.nii.gz', 'pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t09_overlap_16_overlap-mode_hann.nii.gz', 'pred_Bou

In [13]:
import numpy as np
import pydicom
from pydicom.encaps import encapsulate
from pydicom import uid
from pathlib import Path
from datetime import datetime
import SimpleITK as sitk
import warnings


In [14]:

# Create output directory
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Filter catalog for magnitude images only (tag_0x0043_0x1030 == 2)
magnitude_catalog = dicom_catalog[dicom_catalog['tag_0x0043_0x1030'] == 2].copy()
print(f"Found {len(magnitude_catalog)} magnitude DICOM files in catalog")
print(f"Timepoints in catalog: {sorted(magnitude_catalog['time_index'].unique())}")


Found 2800 magnitude DICOM files in catalog
Timepoints in catalog: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [15]:

# Get all prediction files
prediction_files = sorted(Path(prediction_path).glob("*.nii.gz"))
print(f"\nFound {len(prediction_files)} prediction files")



Found 20 prediction files


In [16]:

# Function to resample 3D prediction to DICOM space
def resample_prediction_to_dicom_space(prediction_nifti_path, dicom_filepaths):
    """
    Resample 3D prediction from NIfTI space (RAS) to DICOM space (LPS).
    
    Args:
        prediction_nifti_path: Path to prediction NIfTI file (3D volume)
        dicom_filepaths: List of DICOM file paths for this timepoint (sorted by slice)
    
    Returns:
        3D numpy array in DICOM space [Z, Y, X]
    """
    # Load prediction (3D volume in RAS space)
    pred_img = sitk.ReadImage(str(prediction_nifti_path))
    
    # Load all DICOMs for this timepoint into a 3D volume (LPS space)
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(fp) for fp in dicom_filepaths])
    dicom_3d = reader.Execute()  # 3D volume in LPS space
    
    # Resample prediction to match DICOM 3D space
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(dicom_3d)  # Reference is in LPS
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(0.0)
    
    resampled_3d = resampler.Execute(pred_img)  # Resampled from RAS to LPS
    
    # Convert to numpy array [Z, Y, X] format
    resampled_array = sitk.GetArrayFromImage(resampled_3d)
    
    return resampled_array


In [17]:

# Generate new SeriesInstanceUID for all files (same for all timepoints)
new_series_instance_uid = uid.generate_uid()
current_datetime = datetime.now()
content_date = current_datetime.strftime('%Y%m%d')
content_time = current_datetime.strftime('%H%M%S.%f')[:-3]  # Include milliseconds

print(new_series_instance_uid)

1.2.826.0.1.3680043.8.498.49797055692235613136110042800610303478


In [18]:

# Process each timepoint
for timepoint in sorted(magnitude_catalog['time_index'].unique()):
    print(f"\n{'='*60}")
    print(f"Processing timepoint {timepoint}")
    print(f"{'='*60}")
    
    # Find prediction file for this timepoint
    # Pattern: *_t{timepoint:02d}_*.nii.gz
    pred_pattern = f"*_t{timepoint:02d}_*.nii.gz"
    pred_files = list(Path(prediction_path).glob(pred_pattern))
    
    if not pred_files:
        print(f"  WARNING: No prediction file found for timepoint {timepoint}")
        continue
    
    if len(pred_files) > 1:
        print(f"  WARNING: Multiple prediction files found, using first: {pred_files[0]}")
    
    prediction_file = pred_files[0]
    print(f"  Using prediction file: {prediction_file.name}")
    
    # Filter catalog for this timepoint - magnitude only
    catalog_tp = magnitude_catalog[magnitude_catalog['time_index'] == timepoint].copy()
    
    if len(catalog_tp) == 0:
        print(f"  WARNING: No magnitude DICOMs found for timepoint {timepoint}")
        continue
    
    # Sort by slice_index
    catalog_tp = catalog_tp.sort_values('slice_index').reset_index(drop=True)
    dicom_filepaths = [Path(row['filepath']) for _, row in catalog_tp.iterrows()]
    
    print(f"  Found {len(catalog_tp)} magnitude DICOMs for timepoint {timepoint}")
    
    # Resample entire 3D prediction to DICOM space
    print(f"  Resampling prediction to DICOM space...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resampled_3d = resample_prediction_to_dicom_space(prediction_file, dicom_filepaths)
    
    print(f"  Resampled shape: {resampled_3d.shape} [Z, Y, X]")
    
    # Process each slice
    for idx, row in catalog_tp.iterrows():
        dicom_path = Path(row['filepath'])
        slice_idx = row['slice_index']
        
        if not dicom_path.exists():
            print(f"  WARNING: DICOM file not found: {dicom_path}")
            continue
        
        # Create output path with "_vse" suffix
        dicom_name = dicom_path.stem
        dicom_ext = dicom_path.suffix
        output_path = Path(output_dir) / f"{dicom_name}_vse{dicom_ext}"
        
        try:
            # Load DICOM
            ds = pydicom.dcmread(dicom_path)
            
            # Decompress if necessary
            if hasattr(ds.file_meta, 'TransferSyntaxUID'):
                if ds.file_meta.TransferSyntaxUID.is_compressed:
                    ds.decompress()
            
            # Extract the corresponding slice from resampled 3D volume
            # Use idx (array position) since catalog is sorted by slice_index
            # and array is built from sorted filepaths
            array_idx = idx
            if array_idx >= resampled_3d.shape[0]:
                print(f"  WARNING: Array index {array_idx} >= volume depth {resampled_3d.shape[0]}, using last slice")
                array_idx = resampled_3d.shape[0] - 1
            
            resampled_pred = resampled_3d[array_idx, :, :]  # Extract [Y, X] slice
            
            # Scale from normalized [0, 1] range back to uint16 [0, 65535]
            # Assuming predictions are in [0, 1] range
            resampled_pred = resampled_pred * 65535.0
            resampled_pred = np.clip(resampled_pred, 0, 65535)
            resampled_pred_uint16 = resampled_pred.astype(np.uint16)
            
            # Update metadata
            ds.PatientID = patient_id
            ds.SeriesInstanceUID = new_series_instance_uid
            ds.SOPInstanceUID = uid.generate_uid()
            
            # Update SeriesNumber
            original_series_number = getattr(ds, 'SeriesNumber', 1)
            if isinstance(original_series_number, (int, str)):
                try:
                    original_num = int(original_series_number)
                    ds.SeriesNumber = 9000 + (original_num % 100)
                except (ValueError, TypeError):
                    ds.SeriesNumber = 9000
            else:
                ds.SeriesNumber = 9000
            
            # Update SeriesDescription
            original_description = getattr(ds, 'SeriesDescription', '')
            if original_description:
                ds.SeriesDescription = f"{original_description} VSE"
            else:
                ds.SeriesDescription = "Magnitude VSE Prediction"
            
            # Update date/time tags
            ds.ContentDate = content_date
            ds.ContentTime = content_time
            if hasattr(ds, 'SeriesDate'):
                ds.SeriesDate = content_date
            if hasattr(ds, 'SeriesTime'):
                ds.SeriesTime = content_time
            
            # Store original transfer syntax to preserve compression format
            original_transfer_syntax = None
            if hasattr(ds.file_meta, 'TransferSyntaxUID'):
                original_transfer_syntax = ds.file_meta.TransferSyntaxUID
            
            # Replace pixel data
            if original_transfer_syntax and original_transfer_syntax.is_compressed:
                if 'RLE' in str(original_transfer_syntax):
                    ds.PixelData = encapsulate(resampled_pred_uint16.tobytes())
                else:
                    print(f"  WARNING: Original DICOM uses {original_transfer_syntax} compression. Saving as uncompressed.")
                    ds.PixelData = resampled_pred_uint16.tobytes()
                    ds.file_meta.TransferSyntaxUID = uid.ExplicitVRLittleEndian
            else:
                ds.PixelData = resampled_pred_uint16.tobytes()
            
            # Update pixel data related tags for unsigned 16-bit integer
            ds.BitsAllocated = 16
            ds.BitsStored = 16
            ds.HighBit = 15
            ds.PixelRepresentation = 0  # Unsigned integer (0-65535)
            
            # Save modified DICOM
            ds.save_as(output_path, write_like_original=False)
            
            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{len(catalog_tp)} slices...")
            
        except Exception as e:
            print(f"  ERROR processing DICOM {dicom_path.name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"  Completed timepoint {timepoint}")

print(f"\n{'='*60}")
print(f"Conversion complete! Output saved to: {output_dir}")
print(f"{'='*60}")


Processing timepoint 0
  Using prediction file: pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t00_overlap_16_overlap-mode_hann.nii.gz
  Found 140 magnitude DICOMs for timepoint 0
  Resampling prediction to DICOM space...
  Resampled shape: (140, 256, 256) [Z, Y, X]
  Processed 10/140 slices...
  Processed 20/140 slices...
  Processed 30/140 slices...
  Processed 40/140 slices...
  Processed 50/140 slices...
  Processed 60/140 slices...
  Processed 70/140 slices...
  Processed 80/140 slices...
  Processed 90/140 slices...
  Processed 100/140 slices...
  Processed 110/140 slices...
  Processed 120/140 slices...
  Processed 130/140 slices...
  Processed 140/140 slices...
  Completed timepoint 0

Processing timepoint 1
  Using prediction file: pred_Boudubat_devoted-frost-50_epoch-630_all_timepoints_t01_overlap_16_overlap-mode_hann.nii.gz
  Found 140 magnitude DICOMs for timepoint 1
  Resampling prediction to DICOM space...
  Resampled shape: (140, 256, 256) [Z, Y, X]
  Processed